In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
from scipy.ndimage import gaussian_filter
from Functions import read_ir, thermoelasticity, weighted_median, mmtri_with_history, plot_weight_evolution, compute_ism, interpolate_to_match  # <-- Ensure this module is accessible
from scipy.ndimage import gaussian_filter
from matplotlib.ticker import ScalarFormatter
from matplotlib.colors import to_rgba

plt.rcParams.update({"font.family":"serif","font.sans-serif":["Computer Modern"]})

In [ ]:
# ----------------------
# User-provided data loading
# ----------------------

path = r"Data_preliminary/181_3_hz/*.hcc" # Path to data
filenames = glob.glob(path)

# Read thermal data
data, _ = read_ir(filenames)

# Convert Kelvin to Celsius
data = data - 273.15

foi = 117.9# Hz
fs = 1000  # Sampling frequency in Hz

segments = 4
overlap = 0.25

n_samples = data.shape[0]  # assuming first dimension is time
segment_length = int(n_samples / segments)

fft, freq = thermoelasticity(data, fs, method="fft", segment_length=segment_length, overlap=overlap)
foi_index = np.argmin(np.abs(freq - foi))
amplitude = fft[foi_index]

In [ ]:
# amplitudes = [amplitude,...] 
# reference = []
V = np.stack(amplitudes, axis=0)

In [ ]:
# Run MMTRI with history tracking
filtered_map, final_weights, variance_map, residual_map, history = \
    mmtri_with_history(V, ground_truth=reference, 
                      max_iter=10, convergence_threshold=0.99)

In [ ]:
# Plot Figure 1: Weight evolution
condition_labels = ['Peak solar', 'High solar', 'Moderate solar', 
                   'Twilight', 'Shadow', 'Night']

plot_weight_evolution(history['weights'], condition_labels)